data cleaning pipeline


In [1]:
import pandas as pd
import numpy as np
import json


In [2]:
df = pd.read_csv('../data/raw/nyc_parking_tickets_sample.csv', dtype=str)
df.columns = df.columns.str.replace(' ', '_')


In [3]:
df['Issue_Date'] = pd.to_datetime(df['Issue_Date'], format='%m/%d/%Y', errors='coerce')
df['Month'] = df['Issue_Date'].dt.month
df['Month_Name'] = df['Issue_Date'].dt.month_name()
df['Quarter'] = df['Issue_Date'].dt.quarter
df['Day_of_Week'] = df['Issue_Date'].dt.day_name()
df['Day_Num'] = df['Issue_Date'].dt.dayofweek
df['Week_Num'] = df['Issue_Date'].dt.isocalendar().week
df['Is_Weekend'] = df['Day_Num'].isin([5, 6]).astype(int)


In [4]:
df['Fine_Amount'] = pd.to_numeric(df['Fine_Amount'], errors='coerce')
df.loc[df['Fine_Amount'] <= 0, 'Fine_Amount'] = np.nan
df['Fine_Amount'] = df['Fine_Amount'].fillna(df.groupby('Violation_Description')['Fine_Amount'].transform('median'))
df['Fine_Category'] = pd.cut(df['Fine_Amount'], bins=[0, 50, 100, float('inf')], labels=['Low', 'Medium', 'High'])


In [5]:
df['Vehicle_Year'] = pd.to_numeric(df['Vehicle_Year'], errors='coerce')
df.loc[(df['Vehicle_Year'] == 9999) | (df['Vehicle_Year'] > 2014) | (df['Vehicle_Year'] < 1980), 'Vehicle_Year'] = np.nan
df['Vehicle_Age'] = 2014 - df['Vehicle_Year']
df['Vehicle_Age_Group'] = pd.cut(df['Vehicle_Age'], bins=[-1, 5, 10, 15, float('inf')], labels=['0-5', '6-10', '11-15', '16+'])


In [6]:
color_map = {'GY': 'GRAY', 'BK': 'BLACK', 'WH': 'WHITE'}
df['Vehicle_Color'] = df['Vehicle_Color'].str.upper().replace(color_map)


In [7]:
df['Violation_County'] = df['Violation_County'].str.title()


In [8]:
top_makes = df['Vehicle_Make'].value_counts().nlargest(15).index
df['Vehicle_Make'] = np.where(df['Vehicle_Make'].isin(top_makes), df['Vehicle_Make'], 'OTHER')


In [9]:
passenger = ['SUBN', '4DSD', '2DSD']
commercial = ['VAN', 'DELV', 'PICK']
df['Vehicle_Type_Group'] = 'OTHER'
df.loc[df['Vehicle_Body_Type'].isin(passenger), 'Vehicle_Type_Group'] = 'Passenger'
df.loc[df['Vehicle_Body_Type'].isin(commercial), 'Vehicle_Type_Group'] = 'Commercial'


In [10]:
valid_states = ['NY', 'NJ', 'PA', 'CT', 'FL', 'MA', 'TX', 'VA', 'MD', 'NC']
df.loc[~df['Registration_State'].isin(valid_states), 'Registration_State'] = 'UNKNOWN'
df['Is_OutOfState'] = (df['Registration_State'] != 'NY').astype(int)
df['State_Group'] = np.where(df['Registration_State'] == 'NY', 'NY', np.where(df['Registration_State'].isin(['NJ', 'CT']), 'Tri-State', 'Other'))


In [11]:
df['Plate_Category'] = np.where(df['Plate_Type'] == 'PAS', 'Passenger', 'Other')


In [12]:
df['Street_Name'] = df['Street_Name'].str.upper().str.strip()
df['Street_Type'] = df['Street_Name'].str.split().str[-1]


In [13]:
street_stats = df.groupby('Street_Name').agg(
    Street_Total_Revenue=('Fine_Amount', 'sum'),
    Street_Ticket_Count=('Summons_Number', 'count')
).reset_index()
df = df.merge(street_stats, on='Street_Name', how='left')


In [14]:
plate_counts = df['Plate_ID'].value_counts()
df['Ticket_Count_Per_Plate'] = df['Plate_ID'].map(plate_counts).fillna(0)
df['Is_Repeat_Offender'] = (df['Ticket_Count_Per_Plate'] > 1).astype(int)
df['Offender_Tier'] = pd.cut(df['Ticket_Count_Per_Plate'], bins=[0, 2, 5, float('inf')], labels=['1-2', '3-5', '6+'])


In [15]:
safety = ['FIRE HYDRANT', 'DOUBLE PARKING', 'CROSSWALK']
df['Violation_Severity'] = np.where(df['Violation_Description'].isin(safety), 'Safety-Critical', 'Standard')
avoidable = ['NO PARKING', 'STREET CLEANING', 'EXPIRED MUNI METER', 'NO STANDING']
df['Is_Avoidable'] = df['Violation_Description'].str.contains('|'.join(avoidable), na=False).astype(int)


In [16]:
df['Data_Completeness_Score'] = df.notnull().sum(axis=1)
df['Is_Complete_Record'] = (df['Data_Completeness_Score'] > 20).astype(int)


In [17]:
df = df.drop_duplicates(subset=['Summons_Number'])


In [18]:
transformation_log = {
    "step_01_dates": "3 formats -> unified YYYY-MM-DD",
    "step_02_fines": "Negatives -> NaN, imputed",
    "step_03_years": "outliers -> NaN",
    "step_04_colors": "GY->GRAY etc",
    "step_05_boroughs": "Title Case",
    "step_06_makes": "top 15 kept",
    "step_07_body_type": "10 codes -> groups",
    "step_08_states": "Invalid -> UNKNOWN",
    "step_09_streets": "UPPER + strip",
    "step_10_plates": "Ticket count per plate",
    "step_11_dedup": "Duplicate Summons_Number removed",
    "step_12_validate": "Passed"
}
with open('../docs/etl_transformation_log.json', 'w') as f:
    json.dump(transformation_log, f, indent=2)


In [19]:
df.to_csv('../data/processed/nyc_parking_clean.csv', index=False)
